In [4]:
import os, sys
from pathlib import Path
import json

import pandas as pd

WORK_DIR = Path.cwd().parent

sys.path.append(str(WORK_DIR))

from src import (
    main,
    datasets,
    prompt_formatters as pf,
    response_processing as rp,
    video_tools
)
from src.STAR_utils.visualization_tools import qa_visualization as qaviz



In [ ]:
# File containing the STAR qustions and answers
STAR_VAL_SMALL = WORK_DIR / "data/datasets/STAR/STAR_annotations/STAR_val_small_1000.json"
# File with the generated graphs associated to the above STAR set
SGG_GRAPHS = "/megaverse/storage/lusha/sgg/aggregated_final_sgg_gemini2.5flash_1000_OutTokens8192_20250829_22:00:00.jsonl"

Load the SGG data and show how one instance is structured:

In [6]:
with open(SGG_GRAPHS, 'r') as f:
    sgg_data = [json.loads(line) for line in f]

sgg_data[0]

{'key': 'Feasibility_T2_860',
 'request': {'contents': [{'role': 'user',
    'parts': [{'text': 'You are an intelligent video comprehension model and are going to receive as input a sequence of images extracted from a video. You need to analyze and describe the scene unfolding in the video (sequence of frames) following these guidelines:\n1. Look for recurring objects;\n2. Note that the same object may appear differently across frames due to low image quality, varying viewing angles, or partial obstructions. Carefully check objects with similar features (e.g. shape, color), appearing in different frames, as they may still be the same object;\n3. Pay attention to how the person interacts with its environment\n4. Understand the directional movement of the people and objects in the video\n5. Carefully analyze the chronological sequence of the events occurring in the video;\n6. Pay attention to the atomic and fine grained movement, pose and actions of the people in the video. Compose these

`key` contains the id of the instance of the STAR to which the graph refers to
sgg_data[0]['key']

`stsg` contains the extracted Scene Graph, ready to be given in input to the Graph-QA pipeline
sgg_data[0]['graph']

`request` contains the structure object used to make the requetst at Gemini API
sgg_data[0]['request']

In [23]:
sgg_data[0]['request'].keys()

dict_keys(['contents', 'generationConfig'])

The key `contents` in `request` is a list of `prompt` objects, one for each turn of the conversation

#### the first element the first turn - user question

In [ ]:
sgg_data[0]['request']['contents'][2]

{'role': 'user',
 'parts': [{'text': 'Now organize the objects and relationships you identified before into a formal spatio-temporal scene graph using this format for the predicates: \nobject1 ---- relationship ---- object2\n\nThe spatio-temporal scene graph should be enclosed between the tags <stsg> and </stsg>.\nThe list of relationship predicates pertaining a frame should be introduced by the frame ID or name and enclosed in the tags <scene_graph> and </scene_graph>:\n\n\nFor example:\n<stsg>\nFrame 0:\n<scene_graph>\nman ----  sitting_on ---- chair\ndog ---- lying_under ----  table\nbook ---- on_top_of ---- shelf\nwoman ---- on_the_right_of ---- man\n... \n</scene_graph>\n\nFrame 1:\n<scene_graph>\nwoman ---- on_the_left_of ---- man\n...\n</scene_graph>\n</stsg>\n\n\nPlease follow these guidelines:\n1. Create an appropriate number of relationship triplets (more if the image is complex)\n2. Use specific and consistent object labels\n3. Use concise but descriptive relationship terms 

In [25]:
print(sgg_data[0]['request']['contents'][0]['parts'][0]['text'])

You are an intelligent video comprehension model and are going to receive as input a sequence of images extracted from a video. You need to analyze and describe the scene unfolding in the video (sequence of frames) following these guidelines:
1. Look for recurring objects;
2. Note that the same object may appear differently across frames due to low image quality, varying viewing angles, or partial obstructions. Carefully check objects with similar features (e.g. shape, color), appearing in different frames, as they may still be the same object;
3. Pay attention to how the person interacts with its environment
4. Understand the directional movement of the people and objects in the video
5. Carefully analyze the chronological sequence of the events occurring in the video;
6. Pay attention to the atomic and fine grained movement, pose and actions of the people in the video. Compose these atomic actions happening across the frame to infer the higher level action performed by the person;
7.

In [16]:
print(sgg_data[0]['request']['contents'][0]['parts'][1])

{'inline_data': {'mime_type': 'image/png', 'data': '...'}}


In [17]:
print(sgg_data[0]['request']['contents'][0]['parts'][2])

{'text': '\n\nImage 1:\n'}


In [18]:
print(sgg_data[0]['request']['contents'][0]['parts'][3])

{'inline_data': {'mime_type': 'image/png', 'data': '...'}}


And so on... With all the frames for the video.

#### the second element is the response of the model, which contains the description of the video

In [29]:
print(sgg_data[0]['request']['contents'][1]['parts'][0]['text'])

The video sequence shows a person interacting with an object on a wall, while holding a book.

In **Frame 0**, a person wearing a green t-shirt is visible on the right side of the frame, facing towards the left. They are holding an open book in their left hand, near their waist. To their left, there's a window offering a view of an outdoor building structure. To their right, a doorway leads into a dimly lit room with ornate wooden screen visible in the background.

In **Frame 1**, the person slightly raises their right hand towards the wall in front of them, while still holding the book in their left hand. Their gaze is directed towards the wall.

In **Frame 2**, the person's right hand is now touching a small, rectangular, wooden object mounted on the wall. The object appears to be a small box or shelf. The person is looking intently at this object.

In **Frame 3**, the person continues to touch the wooden object with their right hand. Their head is now slightly tilted upwards, and th

#### The third element is the reply from the user, asking to structure the descritpion into a Scene Graph

In [30]:
print(sgg_data[0]['request']['contents'][2]['parts'][0]['text'])

Now organize the objects and relationships you identified before into a formal spatio-temporal scene graph using this format for the predicates: 
object1 ---- relationship ---- object2

The spatio-temporal scene graph should be enclosed between the tags <stsg> and </stsg>.
The list of relationship predicates pertaining a frame should be introduced by the frame ID or name and enclosed in the tags <scene_graph> and </scene_graph>:


For example:
<stsg>
Frame 0:
<scene_graph>
man ----  sitting_on ---- chair
dog ---- lying_under ----  table
book ---- on_top_of ---- shelf
woman ---- on_the_right_of ---- man
... 
</scene_graph>

Frame 1:
<scene_graph>
woman ---- on_the_left_of ---- man
...
</scene_graph>
</stsg>


Please follow these guidelines:
1. Create an appropriate number of relationship triplets (more if the image is complex)
2. Use specific and consistent object labels
3. Use concise but descriptive relationship terms (connect words with underscores)
4. Include all meaningful relation

#### The 4th element is the response from the model to the last request

In [31]:
print(sgg_data[0]['request']['contents'][3]['parts'][0]['text'])

<stsg>
Frame 0:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
person ---- located_near ---- window
window ---- showing ---- outdoor_building_structure
doorway ---- leading_to ---- dimly_lit_room
room ---- containing ---- ornate_wooden_screen
</scene_graph>

Frame 1:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
person ---- raising ---- right_hand
right_hand ---- moving_towards ---- wall
person ---- looking_at ---- wall
</scene_graph>

Frame 2:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
right_hand ---- touching ---- wooden_object
wooden_object ---- mounted_on ---- wall
person ---- looking_at ---- wooden_object
</scene_graph>

Frame 3:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
right_hand ---- touching ---- wooden_object
person ---- looking_at ---- wooden_object
another_person ---- partially_visible_in ---- backgroun

In [33]:
# which can be accesssed also through
print(sgg_data[0]['stsg'])

<stsg>
Frame 0:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
person ---- located_near ---- window
window ---- showing ---- outdoor_building_structure
doorway ---- leading_to ---- dimly_lit_room
room ---- containing ---- ornate_wooden_screen
</scene_graph>

Frame 1:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
person ---- raising ---- right_hand
right_hand ---- moving_towards ---- wall
person ---- looking_at ---- wall
</scene_graph>

Frame 2:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
right_hand ---- touching ---- wooden_object
wooden_object ---- mounted_on ---- wall
person ---- looking_at ---- wooden_object
</scene_graph>

Frame 3:
<scene_graph>
person ---- wearing ---- green_t-shirt
person ---- holding ---- open_book
right_hand ---- touching ---- wooden_object
person ---- looking_at ---- wooden_object
another_person ---- partially_visible_in ---- backgroun